In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a1a",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#eee",
    "xtick.color":      "#aaa",
    "ytick.color":      "#aaa",
    "text.color":       "#eee",
    "grid.color":       "#333",
    "font.family":      "monospace",
})
ACCENT  = "#00e5ff"
ACCENT2 = "#ff4081"

# LOAD DATA

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv", encoding="latin")
print(f"\n[DATA] Shape: {df.shape}")
print(df.head(3))


# EDA

# Drop customer ID (not predictive)
df.drop(columns=["customerID"], inplace=True, errors="ignore")

# Fix TotalCharges — it's stored as object with spaces
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Target encode
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Drop rows where target or TotalCharges is null
df.dropna(subset=["Churn", "TotalCharges"], inplace=True)

# Separate features
cat_cols = df.select_dtypes(include="object").columns.tolist()
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
num_cols.remove("Churn")

print(f"\n[PREPROCESSING] Categorical cols : {len(cat_cols)}")
print(f"[PREPROCESSING] Numerical cols   : {len(num_cols)}")

# Label-encode all categorical columns
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Features & target
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Train / test split — stratified to preserve churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n[SPLIT] Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"[SPLIT] Churn rate (train): {y_train.mean():.2%}")


# BASELINE DECISION TREE

baseline = DecisionTreeClassifier(random_state=42)
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

print("BASELINE Decision Tree (no tuning)")
print(f"  Train Accuracy : {baseline.score(X_train, y_train):.4f}  ← likely overfitting")
print(f"  Test  Accuracy : {accuracy_score(y_test, base_pred):.4f}")

# HYPERPARAMETER TUNING
param_grid = {
    "max_depth":        [3, 5, 7, 10, None],
    "min_samples_leaf": [1, 5, 10, 20],
    "criterion":        ["gini", "entropy"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)

best_dt = grid_search.best_estimator_
print("TUNED Decision Tree")
print(f"  Best params    : {grid_search.best_params_}")
print(f"  CV ROC-AUC     : {grid_search.best_score_:.4f}")

y_pred      = best_dt.predict(X_test)
y_prob      = best_dt.predict_proba(X_test)[:, 1]

print(f"  Test Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"  Test ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

# MODEL COMPARISON
# Scale for Logistic Regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

models = {
    "Decision Tree (tuned)": best_dt,
    "Random Forest":         RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Logistic Regression":   LogisticRegression(max_iter=1000, random_state=42),
}

results = {}
for name, model in models.items():
    Xtr = X_train_s if name == "Logistic Regression" else X_train
    Xte = X_test_s  if name == "Logistic Regression" else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    probs = model.predict_proba(Xte)[:, 1]
    results[name] = {
        "Accuracy":  accuracy_score(y_test, preds),
        "ROC-AUC":   roc_auc_score(y_test, probs),
        "model":     model,
        "probs":     probs,
    }

print("MODEL COMPARISON")
print(f"{'Model':<30} {'Accuracy':>10} {'ROC-AUC':>10}")
for name, r in results.items():
    print(f"{name:<30} {r['Accuracy']:>10.4f} {r['ROC-AUC']:>10.4f}")

# VISUALISATIONS

fig = plt.figure(figsize=(20, 22))
fig.suptitle("Telco Churn Prediction Model Analysis", fontsize=18,
             color=ACCENT, fontweight="bold", y=0.98)

# Churn distribution
ax1 = fig.add_subplot(4, 3, 1)
counts = y.value_counts()
bars = ax1.bar(["No Churn", "Churn"], counts.values,
               color=[ACCENT, ACCENT2], width=0.5, edgecolor="none")
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
             f"{val}\n({val/len(y):.1%})", ha="center", fontsize=9, color="#eee")
ax1.set_title("Target Distribution", color=ACCENT, fontsize=11)
ax1.set_ylabel("Customers")
ax1.grid(axis="y", alpha=0.3)

# Tenure vs Churn
ax2 = fig.add_subplot(4, 3, 2)
for label, colour in [(0, ACCENT), (1, ACCENT2)]:
    ax2.hist(df.loc[df["Churn"] == label, "tenure"],
             bins=30, alpha=0.7, color=colour,
             label=["No Churn", "Churn"][label], edgecolor="none")
ax2.set_title("Tenure Distribution by Churn", color=ACCENT, fontsize=11)
ax2.set_xlabel("Tenure (months)")
ax2.legend()
ax2.grid(alpha=0.3)

# Monthly charges vs Churn
ax3 = fig.add_subplot(4, 3, 3)
for label, colour in [(0, ACCENT), (1, ACCENT2)]:
    ax3.hist(df.loc[df["Churn"] == label, "MonthlyCharges"],
             bins=30, alpha=0.7, color=colour,
             label=["No Churn", "Churn"][label], edgecolor="none")
ax3.set_title("Monthly Charges by Churn", color=ACCENT, fontsize=11)
ax3.set_xlabel("Monthly Charges ($)")
ax3.legend()
ax3.grid(alpha=0.3)

# ROC Curves
ax4 = fig.add_subplot(4, 3, 4)
colours = [ACCENT, ACCENT2, "#ffd740"]
for (name, r), col in zip(results.items(), colours):
    fpr, tpr, _ = roc_curve(y_test, r["probs"])
    ax4.plot(fpr, tpr, color=col, linewidth=2,
             label=f"{name.split('(')[0].strip()} (AUC={r['ROC-AUC']:.3f})")
ax4.plot([0, 1], [0, 1], "--", color="#555", linewidth=1)
ax4.set_title("ROC Curves — All Models", color=ACCENT, fontsize=11)
ax4.set_xlabel("False Positive Rate")
ax4.set_ylabel("True Positive Rate")
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

# Confusion matrix
ax5 = fig.add_subplot(4, 3, 5)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax5,
            linewidths=0.5, linecolor="#333",
            xticklabels=["No Churn", "Churn"],
            yticklabels=["No Churn", "Churn"])
ax5.set_title("Confusion Matrix (Tuned DT)", color=ACCENT, fontsize=11)
ax5.set_ylabel("Actual")
ax5.set_xlabel("Predicted")

# Model comparison bar
ax6 = fig.add_subplot(4, 3, 6)
model_names   = [n.split("(")[0].strip() for n in results]
accuracies    = [r["Accuracy"] for r in results.values()]
roc_aucs      = [r["ROC-AUC"]  for r in results.values()]
x = np.arange(len(model_names))
w = 0.35
ax6.bar(x - w/2, accuracies, w, color=ACCENT,  label="Accuracy",  edgecolor="none")
ax6.bar(x + w/2, roc_aucs,   w, color=ACCENT2, label="ROC-AUC",   edgecolor="none")
ax6.set_xticks(x)
ax6.set_xticklabels(model_names, fontsize=8, rotation=10)
ax6.set_ylim(0.5, 1.0)
ax6.set_title("Model Comparison", color=ACCENT, fontsize=11)
ax6.legend()
ax6.grid(axis="y", alpha=0.3)

#Feature importance
ax7 = fig.add_subplot(4, 3, (7, 9))
rf_model    = results["Random Forest"]["model"]
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top15       = importances.nlargest(15).sort_values()
colours_bar = [ACCENT2 if i >= 12 else ACCENT for i in range(len(top15))]
top15.plot(kind="barh", ax=ax7, color=colours_bar, edgecolor="none")
ax7.set_title("Top 15 Feature Importances (Random Forest)", color=ACCENT, fontsize=11)
ax7.set_xlabel("Importance")
ax7.grid(axis="x", alpha=0.3)
for i, val in enumerate(top15):
    ax7.text(val + 0.001, i, f"{val:.3f}", va="center", fontsize=8)

# Decision tree visualisation
ax8 = fig.add_subplot(4, 3, (10, 12))
shallow_dt = DecisionTreeClassifier(max_depth=3, random_state=42)
shallow_dt.fit(X_train, y_train)
plot_tree(
    shallow_dt,
    feature_names=X.columns.tolist(),
    class_names=["No Churn", "Churn"],
    filled=True,
    rounded=True,
    fontsize=7,
    ax=ax8,
)
ax8.set_title("Decision Tree Structure (depth=3)", color=ACCENT, fontsize=11)

plt.savefig("churn_analysis.png", dpi=150, bbox_inches="tight", facecolor="#0f0f0f")
plt.close()
print("\n[PLOT] Saved → churn_analysis.png")

# BUSINESS INSIGHTS
print("  BUSINESS INSIGHTS")

top5 = importances.nlargest(5)
print("\nTop 5 Churn Drivers:")
for feat, score in top5.items():
    print(f"  {feat:<25} importance = {score:.4f}")

high_risk = df[(df["tenure"] < 12) & (df["MonthlyCharges"] > 65)]
print(f"\nHigh-Risk Segment (tenure < 12 months & charges > $65):")
print(f"  Customers : {len(high_risk):,}")
print(f"  Churn rate: {high_risk['Churn'].mean():.2%}  (vs {y.mean():.2%} overall)")






[DATA] Shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No              No  Month-to-month              Yes   
1          No          No              No        One year               No   
2          No          No              No  Month-to-month              Yes   

      PaymentMethod MonthlyCharges 